In [ ]:
# ============================
# 1. 기본 세팅
# ============================
!pip install openai tqdm -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import numpy as np
from tqdm import tqdm
from openai import OpenAI
import os
from typing import List

# Upstage API Key 입력
UPSTAGE_API_KEY = "your-api-key"

client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1"
)

# 사용할 임베딩 모델
EMBED_MODEL = "embedding-passage"

# 파일 경로
MMLU_JSONL_PATH = "/content/drive/MyDrive/nlp_me/data/wiki_200topics.jsonl"
OUTPUT_NPZ_PATH = "/content/drive/MyDrive/nlp_me/result/result_mmlu/mmlu_wiki_embeddings.npz"

# 폴더 자동 생성
os.makedirs(os.path.dirname(OUTPUT_NPZ_PATH), exist_ok=True)


In [ ]:
# ============================
# 2. JSONL 파일 로드
# ============================

def load_jsonl_texts(file_path: str, text_key: str = "text") -> List[str]:
    """
    jsonl 파일을 한 줄씩 읽어 text_key 필드의 내용을 리스트로 반환
    """
    texts = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                if text_key in obj and isinstance(obj[text_key], str):
                    texts.append(obj[text_key].strip())
            except json.JSONDecodeError:
                continue
    return texts

texts = load_jsonl_texts(MMLU_JSONL_PATH, text_key="text")
print(f"총 {len(texts)}개 문서 로드 완료.")
print("예시 문서:\n", texts[0][:500])


총 5208개 문서 로드 완료.
예시 문서:
 A constitution, or supreme law, is the aggregate of fundamental principles or established precedents that constitute the legal basis of a polity, organization or other type of entity, and commonly determines how that entity is to be governed.
When these principles are written down into a single document or set of legal documents, those documents may be said to embody a written constitution; if they are encompassed in a single comprehensive document, it is said to embody a codified constitution. 


In [ ]:
# ============================
# 3. 배치 단위 임베딩 함수
# ============================

def embed_texts_batch(
    texts: List[str],
    model: str = EMBED_MODEL,
    batch_size: int = 16
) -> np.ndarray:
    """
    여러 텍스트를 배치로 나눠 Upstage 임베딩 API 호출.
    결과 shape = (len(texts), d)
    """
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i : i + batch_size]
        resp = client.embeddings.create(
            model=model,
            input=batch
        )
        for item in resp.data:
            all_embeddings.append(item.embedding)

    emb_array = np.array(all_embeddings, dtype="float32")
    return emb_array


In [ ]:
# ============================
# 4. MMLU 위키 임베딩 생성
# ============================

mmlu_embeddings = embed_texts_batch(texts, model=EMBED_MODEL, batch_size=16)
print("임베딩 shape:", mmlu_embeddings.shape)


100%|██████████| 326/326 [07:59<00:00,  1.47s/it]


임베딩 shape: (5208, 4096)


In [ ]:
# ============================
# 5. npz로 저장 (embeddings + chunks)
# ============================

def save_kb_npz(embeddings: np.ndarray, chunks: List[str], out_path: str):
    """
    embeddings, chunks를 npz 형식으로 저장
    - embeddings: (N, d) float32
    - chunks: (N,) object / str
    """
    np.savez(out_path, embeddings=embeddings, chunks=np.array(chunks, dtype=object))
    print(f"Saved to {out_path}")
    print(" - embeddings:", embeddings.shape)
    print(" - chunks:", len(chunks))

save_kb_npz(mmlu_embeddings, texts, OUTPUT_NPZ_PATH)


✅ Saved to /content/drive/MyDrive/nlp_me/result/result_mmlu/mmlu_wiki_embeddings.npz
 - embeddings: (5208, 4096)
 - chunks: 5208


In [ ]:
# ============================
# 6. 테스트 로드
# ============================

data = np.load(OUTPUT_NPZ_PATH, allow_pickle=True)
print("로드된 임베딩 shape:", data["embeddings"].shape)
print("텍스트 청크 수:", len(data["chunks"]))
print("예시 청크:", data["chunks"][0][:300])


🔍 로드된 임베딩 shape: (5208, 4096)
🔍 텍스트 청크 수: 5208
예시 청크: A constitution, or supreme law, is the aggregate of fundamental principles or established precedents that constitute the legal basis of a polity, organization or other type of entity, and commonly determines how that entity is to be governed.
When these principles are written down into a single docu
